In [ ]:
# --timeframe 1d   : Таймфрейм свечей (дневные данные).
# --start-year 2000: Глубина загрузки истории (начиная с 2000 года).
# --workers 6      : Количество параллельных потоков для ускорения загрузки.

!python -m _tools.update_market_data --start-year 2020 --timeframe 5m --workers 4
!python -m _tools.update_macro --start-year 2020 --timeframe 5m --workers 4
# Выполняет комплексную проверку целостности, отсутствия пропусков и корректности OHLCV данных.
!python -m _tools.check_data_quality --timeframe 5m

📊 Чтение списка инструментов из universe.csv...
✅ Найдено инструментов: 31 (binance: 30, coinex: 1)
🚀 Запуск (CCXT Async) | Таймфрейм = 5m | Год старта = 2020 | Потоков = 4
--------------------------------------------------
📥 BTC/USDT (5m) c 2026-07-16:   0%|                 | 0/1794 [00:00<?, ? св/s]

📥 ETH/USDT (5m) c 2026-07-16:   0%|                 | 0/1794 [00:00<?, ? св/s]


📥 BNB/USDT (5m) c 2026-07-16:   0%|                 | 0/1794 [00:00<?, ? св/s]



📥 SOL/USDT (5m) c 2026-07-16:   0%|                 | 0/1794 [00:00<?, ? св/s]
📥 BTC/USDT (5m) c 2026-07-16:  56%|▌| 1000/1794 [00:03<00:03, 255.03 св/s, по 



📥 SOL/USDT (5m) c 2026-07-16:  56%|▌| 1000/1794 [00:02<00:01, 401.87 св/s, по 

📥 ETH/USDT (5m) c 2026-07-16:  56%|▌| 1000/1794 [00:03<00:02, 283.79 св/s, по 


📥 BNB/USDT (5m) c 2026-07-16:  56%|▌| 1000/1794 [00:03<00:02, 326.56 св/s, по 
📥 BTC/USDT (5m) c 2026-07-16: 1795 св [00:04, 411.33 св/s, по 2026-07-22]     
                                                     

In [ ]:
!bash start_data_swarm.sh

In [ ]:
!bash stop_data_swarm.sh

In [ ]:
!python -m _tools.check_and_heal_data

In [ ]:
#полная проверка подготовленных на предыдущем этапе данных
!python -m _tools.verify_data

In [ ]:
#Запуск нескольких процессов в параллели
!bash start_swarm_amd.sh \
    --dataset_dir "data/processed/2000_2026_1d_18_3" \
    --bonus_ratio 0.2 \
    --min_delta 0.001 \
    --runs 500 --epochs 100 \
    --lr 2e-3 --l2_reg 1e-5 \
    --start_fold "fold_2010" \
    --factor 0.5 --patience 3 \
    --vram 10000/1 --stagger 0 \
    --keep 5 \
    --append \
    --arch attention
    #--arch cnn, conv1d+gru, mlp, attention
    #--track_trajectory
    #--init_pca_coord -148.0 -116.0 --init_pca_radius 20.0 \

In [ ]:
!./stop_swarm.sh
!python -m _tools.clean_lstm_models --keep 5

In [ ]:
!python run_all_ensembles.py --dataset_dir "data/processed/2000_2026_1d_18_3" --max_k 20

In [ ]:
%run _tools/plot_landscape.py --fold data/processed/2000_2026_1d_30_5/fold_2014

In [ ]:
%matplotlib inline
%run _tools/analyze_runs.py data/processed/2000_2026_1d_3_1/fold_2026 --runs 100 --arch attention

In [ ]:
!python -m _tools.prepare_rl_env

In [ ]:
!python -m _tools.check_env_data

In [ ]:
!python -m _tools.find_healthy_checkpoints --target_phase 1 --position end --exp_name pbt_trading_bot_phase2
#--clean

🔍 Поиск чекпоинтов: Стадия 1 | Позиция: end
✅ PPO_TradingEnv-... | Целевая итер: 1162 | Взят чекпоинт: checkpoint_000123
✅ PPO_TradingEnv-... | Целевая итер: 1169 | Взят чекпоинт: checkpoint_000123
✅ PPO_TradingEnv-... | Целевая итер: 1166 | Взят чекпоинт: checkpoint_000121
✅ PPO_TradingEnv-... | Целевая итер: 1150 | Взят чекпоинт: checkpoint_000117
✅ PPO_TradingEnv-... | Целевая итер: 1170 | Взят чекпоинт: checkpoint_000122
✅ PPO_TradingEnv-... | Целевая итер: 1175 | Взят чекпоинт: checkpoint_000134
✅ PPO_TradingEnv-... | Целевая итер: 1156 | Взят чекпоинт: checkpoint_000120
✅ PPO_TradingEnv-... | Целевая итер: 1200 | Взят чекпоинт: checkpoint_000125

💾 Успешно сохранено 8 чекпоинтов в healthy_checkpoints.json


In [ ]:
!python -m _tools.train_rllib_pbt --population 8 --cpu --iterations 3000 --phase2_ratio 0.3 --phase3_ratio 0.7 --start_phase 2 --exp_name pbt_trading_bot_phase2
#--force

In [ ]:
%load_ext tensorboard
%tensorboard --logdir data/processed/2000_2026_1d/rl_env/ray_results
#%tensorboard --logdir "C:\Users\Restorator\Documents\trader_test\trader_test\data\processed\2000_2026_1d\rl_env\ray_results"

In [2]:
!python -m _tools.export_to_git

🔍 Поиск лучшего чекпоинта через разбор OOS метаданных Ray Tune...
⚠️ Валидные чекпоинты не найдены. Возможно, обучение завершилось до первого сохранения.


In [3]:
!python -m _tools.evaluate_agent --checkpoint "/home/restorator/trader_test/data/processed/2000_2026_1d/rl_env/champions/best_model"

I0000 00:00:1781694736.427119   49127 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781694736.429391   49127 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1781694738.751141   49127 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781694738.753223   49127 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
🔄 Восстановление агента из /home/restorator/trader_test/data/processed/2000_2026_1d/rl_env/champions/best_model/policies/default_policy...
/home/restorator/trader_test/.venv-tf/lib/python3.1